<a href="https://colab.research.google.com/github/OpenTopography/STAC-Examples/blob/main/OT_STAC_Interactive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OT Raster STAC Interactive Query Example

This notebook lets users select a bounding box/spatial area of interest on an interactive map, query the OpenTopography Raster STAC Catalog for intersecting datasets in Cloud Optimized GeoTIFF (COG) format, choose a dataset to download, and generate visualizations from the selected data.

Notebook Workflow:

| Step | Purpose |
|------|---------|
| 1 | Install & import dependencies |
| 2 | Configuration |
| 3 | Connect to the OT STAC catalog |
| 4 | Define helper functions |
| 5 | Interactive map — Draw bounding box |
| 6 | Search catalog & select a dataset |
| 7 | View dataset metadata |
| 8 | Generate Color Hillshade |
| 9 | Download output files |
|||

Viswanath Nandigam, Matt Beckley
info@opentopography.org

Work is part of the OpenTopography project supported by NSF Award Numbers 2410799, 2410800 & 2410801


### Step 1 - Install and import dependencies

In [144]:
import subprocess, sys

pkgs = [
    "pystac",         # STAC metadata handling
    "pystac-client",  # STAC API client
    "rasterio",       # Raster data I/O
    "ipyleaflet",     # Interactive maps in Jupyter
    "ipywidgets",     # UI widgets for interactivity
    "matplotlib",     # Plotting and visualization
    "Pillow"          # Image processing
    ]

print("Installing packages …")

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs)

import json, io, base64
from pathlib import Path
from datetime import datetime

import numpy as np
import rasterio
import pystac

import matplotlib.pyplot as plt
from PIL import Image

import ipywidgets as widgets
from ipyleaflet import (
    Map, DrawControl, GeoJSON, ImageOverlay,
    basemaps, LayersControl, ScaleControl
)
from IPython.display import display, HTML, clear_output

print("Done. All dependencies ready.")

Installing packages …
Done. All dependencies ready.


### Step 2 - Configuration

In [145]:
# URL for the OpenTopography raster STAC catalog.
STAC_URL      = "https://portal.opentopography.org/stac/raster_catalog.json"

# Map defaults

# The drawing toolbar is only enabled when the map is zoomed in to this
# level or deeper. This helps avoid very large selections when resources are limited.
MIN_DRAW_ZOOM = 14

# Initial map view shown when the notebook starts. Coordinates are in (latitude, longitude) order.
DEFAULT_CENTER = (37.74, -119.56)  # Yosemite Valley
DEFAULT_ZOOM   = 12
TERRAIN_CMAP   = "terrain"         # matplotlib colormap for color hillshade

# Maximum number of intersecting datasets shown in dropdown
MAX_RESULTS   = 20

# Output directory for generated files
OUTPUT_DIR    = Path("ot_stac_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Shared state used across notebook cells
state = {
    "drawn_bbox"       : None,   # Bounding box from the drawn geometry: [west, south, east, north]
    "search_hits"      : [],     # Intersecting search results as (pystac.Collection, pystac.Item)
    "selected_col"     : None,   # Collection selected by the user
    "selected_item"    : None,   # Item selected by the user
    "hillshade_path"   : None,   # Path to the saved hillshade PNG
    "metadata_path"    : None,   # Path to the saved metadata JSON
    "overlay_layer"    : None,   # Active ImageOverlay currently shown on the map
    "bbox_layer"       : None,   # GeoJSON layer representing the drawn area
    "draw_ctrl_on_map" : False,  # Whether DrawControl is currently on the map
    "current_zoom"     : DEFAULT_ZOOM, # Current map zoom level
}

print(f"Outputs → {OUTPUT_DIR.resolve()}")
print(f"MIN_DRAW_ZOOM = {MIN_DRAW_ZOOM}  (zoom in to enable drawing)")

Outputs → /Users/beckley/Documents/OT/Docs/NewsItems/STAC_NewsItem/ot_stac_outputs
MIN_DRAW_ZOOM = 14  (zoom in to enable drawing)


### Step 3 - Connect to the OT Raster STAC Catalog

In [146]:
#Open the OT Raster STAC catalog (static)
print(f"Connecting to STAC catalog …\n  {STAC_URL}")
catalog = pystac.Catalog.from_file(STAC_URL)
print(f"Done. Connected to STAC Catalog")

# Iterate through the catalog's immediate children and keep only STAC
# collections. This creates a simple in-memory cache for later searches
# and dropdown population in the notebook.
print("Loading collections (walking static JSON links) …")
_collections_cache = [
    child for child in catalog.get_children()
    if isinstance(child, pystac.Collection)
]
print(f"Done. {len(_collections_cache)} collections loaded.")

Connecting to STAC catalog …
  https://portal.opentopography.org/stac/raster_catalog.json
Done. Connected to STAC Catalog
Loading collections (walking static JSON links) …
Done. 283 collections loaded.


### Step 4 - Helper Functions

In [147]:
# ------------------------------------------------------------------------
# Geometry helpers
# ------------------------------------------------------------------------

def bbox_intersects(a, b):
    #Return true if [minx,miny,maxx,maxy] bboxes a and b overlap.
    return not (a[2] <= b[0] or a[0] >= b[2] or a[3] <= b[1] or a[1] >= b[3])

def geojson_rect_to_bbox(geo):
    #Convert a drawn rectangle GeoJSON feature to [west, south, east, north].
    coords = geo["geometry"]["coordinates"][0]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

# ------------------------------------------------------------------------
# STAC asset detection
# ------------------------------------------------------------------------

#Collect all media-type strings from a STAC asset, checking standard attributes and extra_fields.
def _asset_type_strings(asset):
    candidates = []
    # Standard pystac Asset attribute
    candidates.append(getattr(asset, "media_type", "") or "")
    # extra_fields dict (used by some STAC flavours)
    ef = getattr(asset, "extra_fields", {}) or {}
    candidates.append(ef.get("type", "") or "")
    candidates.append(ef.get("media_type", "") or "")
    # The asset dict itself may have a top-level 'type' key
    if hasattr(asset, "to_dict"):
        d = asset.to_dict()
        candidates.append(d.get("type", "") or "")
    return [s.lower() for s in candidates if s]

def is_tiff_asset(asset):
    """
    Return True if an asset is strictly a GeoTIFF/Cloud Optimised GeoTIFF.
    Excludes directory-based legacy formats like Esri Binary Grids (.adf).
    """
    href = (getattr(asset, "href", "") or "").lower()

    # Explicitly skip legacy Esri Binary Grid files
    if href.endswith(".adf") or "/_adf_" in href:
        return False

    for s in _asset_type_strings(asset):
        if "tif" in s:
            return True

    if href.endswith((".tif", ".tiff")):
        return True

    # Cloud-optimised GeoTIFFs often have no extension — also accept any
    # asset whose role or key name suggests raster data
    ef = getattr(asset, "extra_fields", {}) or {}
    roles = ef.get("roles", []) or []
    if any(r in ("data", "overview", "visual") for r in roles):
        # Double check it isn't an adf path hidden without extension
        if ".adf" not in href:
            return True

    return False

def get_asset_bbox(asset, item):
    #Return asset's own bbox if present, otherwise fall back to item bbox.
    ef = getattr(asset, "extra_fields", {}) or {}
    return ef.get("bbox") or item.bbox

def find_tiff_asset(item, query_bbox=None):
    """
    Find the best raster asset for an item using a 2-tier strategy:

      Tier 1 — TIFF by type AND bbox intersects query
      Tier 2 — TIFF by type (any bbox)

    Returns (asset, bbox_list) or (None, None).
    """
    tiff_assets = [(k, a) for k, a in item.assets.items() if is_tiff_asset(a)]

    # Tier 1: TIFF with intersecting bbox
    if query_bbox and tiff_assets:
        for k, a in tiff_assets:
            ab = get_asset_bbox(a, item)
            if ab and bbox_intersects(ab, query_bbox):
                return a, ab

    # Tier 2: Any TIFF asset (use item bbox)
    if tiff_assets:
        k, a = tiff_assets[0]
        return a, get_asset_bbox(a, item)


    return None, None

# ------------------------------------------------------------------------
# STAC search
# ------------------------------------------------------------------------

from concurrent.futures import ThreadPoolExecutor, as_completed

_items_cache = {}  # col.id → [pystac.Item, …]  populated on first fetch

def _check_collection(col, bbox):
    """
    Fetch all items for one collection (cached after first call) and return
    those whose bbox intersects the query bbox.
    Runs in a thread — each collection's HTTP requests are independent.
    """
    if col.id not in _items_cache:
        try:
            _items_cache[col.id] = list(col.get_items())
        except Exception:
            _items_cache[col.id] = []
    return [
        (col, item) for item in _items_cache[col.id]
        if item.bbox and bbox_intersects(item.bbox, bbox)
    ]

# ------------------------------------------------------------------------
# Raster helpers
# ------------------------------------------------------------------------

# Output size bounds for the rendered preview. MAX_DISPLAY_PX is the memory
# safeguard: the array read into RAM is never larger than this in either
# dimension, no matter how big an AOI a user draws. MIN_DISPLAY_PX is a
# display-quality floor: if the AOI window is smaller than this (e.g. a
# tight box over a coarse-resolution global DEM), we gently upsample with
# smooth resampling instead of showing a handful of giant blocky pixels.
MAX_DISPLAY_PX = 1024
MIN_DISPLAY_PX = 512

def read_raster_safe(href, bbox=None, max_px=MAX_DISPLAY_PX, min_px=MIN_DISPLAY_PX):
    """
    Open a raster and return a preview band array covering the area of
    interest — without loading the full dataset into RAM.

    If `bbox` ([west, south, east, north] in EPSG:4326) is given, only the
    pixel window covering that AOI is requested from the remote COG using
    a rasterio windowed read (an HTTP range request under the hood, not a
    full-file download). This is what clips a single global asset like
    GEDTM down to just the drawn AOI instead of returning the whole planet.
    If `bbox` is omitted, the full dataset extent is read instead.

    The output array is resampled to fit between `min_px` and `max_px` in
    its longer dimension: large AOI windows are downsampled with an
    averaging filter (memory safeguard, and avoids aliasing on coarse
    terrain), while small AOI windows on low native-resolution datasets are
    upsampled with bilinear interpolation so the preview isn't a handful of
    blocky pixels. Either way the array held in memory stays small.
    """
    from rasterio.enums import Resampling
    from rasterio.windows import from_bounds
    from rasterio.warp import transform_bounds
    from rasterio.coords import BoundingBox

    with rasterio.open(href) as ds:
        nodata  = ds.nodata
        crs     = ds.crs
        full_tf = ds.transform

        if bbox is not None:
            # Reproject the query bbox into the dataset's own CRS (a no-op
            # for the common case where the dataset is already EPSG:4326),
            # then clip it to the dataset's own bounds so the window never
            # runs past the edges of the raster.
            west, south, east, north = transform_bounds("EPSG:4326", crs, *bbox)
            west  = max(west,  ds.bounds.left)
            south = max(south, ds.bounds.bottom)
            east  = min(east,  ds.bounds.right)
            north = min(north, ds.bounds.top)

            window = from_bounds(west, south, east, north, transform=full_tf)
            window = window.round_offsets().round_lengths()
            transform = ds.window_transform(window)
            bounds = BoundingBox(west, south, east, north)
            src_w = max(1, int(window.width))
            src_h = max(1, int(window.height))
        else:
            # No AOI given — fall back to the full dataset extent.
            window = None
            transform = full_tf
            bounds = ds.bounds
            src_w, src_h = ds.width, ds.height

        # Decide output size and resampling method:
        #   - window too big  → downsample with averaging (memory safeguard)
        #   - window too small → upsample with bilinear (display-quality floor)
        #   - otherwise        → read at native resolution
        if src_w > max_px or src_h > max_px:
            scale = min(max_px / src_w, max_px / src_h)
            resampling = Resampling.average
        elif src_w < min_px or src_h < min_px:
            scale = min(min_px / src_w, min_px / src_h)
            # Defensive cap: never let the upsample push either dimension
            # past max_px (matters for very long, thin AOI rectangles).
            scale = min(scale, max_px / src_w, max_px / src_h)
            resampling = Resampling.bilinear
        else:
            scale = 1.0
            resampling = Resampling.bilinear

        out_w = max(2, int(round(src_w * scale)))
        out_h = max(2, int(round(src_h * scale)))

        band = ds.read(
            1,
            window=window,
            out_shape=(out_h, out_w),
            resampling=resampling,
        ).astype("float32")

        # Guard dimensional sanity: strip arbitrary 3D wrappers if present
        if band.ndim == 3:
            band = band[0]

        # Mask nodata immediately after read
        if nodata is not None:
            band[band == nodata] = np.nan
        # Catch large float sentinels common in Arc/ASCII grids
        band[band < -1e30] = np.nan
        band[band > 1e30]  = np.nan

        # Pixel spacing for the resampled grid (windowing doesn't change the
        # native pixel size — only the out_shape resampling below does)
        if crs and crs.is_geographic:
            mid_lat = (bounds.bottom + bounds.top) / 2.0
            base_dx = abs(transform.a) * 111_320.0 * np.cos(np.deg2rad(mid_lat))
            base_dy = abs(transform.e) * 111_132.0
        else:
            base_dx = abs(transform.a)
            base_dy = abs(transform.e)

        # Scale dx/dy to match the resampled resolution
        dx = base_dx * (src_w / out_w)
        dy = base_dy * (src_h / out_h)

    return band, nodata, dx, dy, bounds, crs, band.shape

# ------------------------------------------------------------------------
# Hillshade & colour rendering
# ------------------------------------------------------------------------

def compute_hillshade(arr, azimuth=315.0, altitude=45.0, dx=1.0, dy=1.0):
    """
    Lambertian hillshade from a 2-D elevation array.
    azimuth is the sun compass bearing (degrees),
    altitude is sun angle above horizon (degrees).
    Returns uint8 array 0–255.
    """
    arr = np.atleast_2d(arr)
    if arr.ndim > 2:
        arr = arr.squeeze()

    # If 1D (e.g., shape was flattened), reconstruct it as a square
    if arr.ndim == 1:
        side = int(np.sqrt(arr.size))
        arr = arr.reshape((side, side))

    nan_mask = np.isnan(arr)
    if nan_mask.any():
        fill = np.nanmean(arr)
        arr[nan_mask] = 0.0 if np.isnan(fill) else fill
    # np.gradient(f, dy, dx) is ambiguous across numpy versions when passing
    # two scalars — use explicit axis keyword instead.
    gy = np.gradient(arr, dy, axis=0)
    gx = np.gradient(arr, dx, axis=1)
    slope  = np.pi / 2.0 - np.arctan(np.hypot(gx, gy))
    aspect = np.arctan2(-gx, gy)
    az, alt = np.deg2rad(azimuth), np.deg2rad(altitude)
    hs = np.sin(alt)*np.sin(slope) + np.cos(alt)*np.cos(slope)*np.cos(az - aspect)
    return (np.clip(hs, 0, 1) * 255).astype(np.uint8)


def make_color_hillshade_rgba(band, nodata=None, cmap_name=TERRAIN_CMAP,
                               azimuth=315.0, altitude=45.0,
                               dx=1.0, dy=1.0, blend=0.6):
    """
      Return a PIL RGBA image blending terrain colour with hillshade relief.
      blend=0 → gives pure greyscale hillshade
      blend=1 → gives pure terrain colour (no relief)
    """
    arr = band.astype("float32")

    # Mask nodata — guard against large sentinel values (e.g. -3.4e+38)
    if nodata is not None:
        arr[arr == nodata] = np.nan
    # Also mask values that are clearly sentinel fill (common in ADF/Arc grids)
    arr[arr < -1e30] = np.nan
    arr[arr > 1e30]  = np.nan

    valid = ~np.isnan(arr)
    if not valid.any():
        # Entire array is nodata — return a fully transparent image
        return Image.fromarray(
            np.zeros((*arr.shape, 4), dtype=np.uint8), mode="RGBA"
        )

    alpha = (valid * 255).astype(np.uint8)

    vmin, vmax = float(np.nanmin(arr)), float(np.nanmax(arr))
    norm = np.where(valid, (arr - vmin) / max(vmax - vmin, 1e-9), 0.0)
    color_rgba = (plt.get_cmap(cmap_name)(norm) * 255).astype(np.uint8)

    hs      = compute_hillshade(arr, azimuth=azimuth, altitude=altitude, dx=dx, dy=dy)
    hs_norm = hs.astype("float32") / 255.0

    rgb     = color_rgba[..., :3].astype("float32") / 255.0
    blended = np.clip(rgb * (blend + (1 - blend) * hs_norm[..., None]), 0, 1)
    return Image.fromarray(
        np.dstack([(blended * 255).astype(np.uint8), alpha]), mode="RGBA"
    )


def pil_to_data_url(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG", optimize=True)
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


print("Done. Helper functions defined.")

Done. Helper functions defined.


### Step 5 - Interactive Map: Draw Bounding Box

In [148]:
# ------------------------------------------------------------------------
# Map setup
# ------------------------------------------------------------------------

m = Map(
    center=DEFAULT_CENTER,
    zoom=DEFAULT_ZOOM,
    scroll_wheel_zoom=True,
    basemap=basemaps.Esri.WorldImagery,
)
m.layout.height = "540px"
m.add_control(LayersControl(position="topright"))
m.add_control(ScaleControl(position="bottomleft"))

# Reset only the bbox-related state each time this cell runs, without
# clobbering keys owned by later steps (search results, selections, overlays).
state["current_zoom"] = DEFAULT_ZOOM
state["draw_ctrl_on_map"] = False
state["drawn_bbox"] = None
state["bbox_layer"] = None

# DrawControl
# Configure a rectangle-only drawing tool. Other geometry types are
# disabled because this workflow expects a rectangular search extent.
# Note: DrawControl is added to the map only when the zoom threshold is met.
draw_ctrl = DrawControl(
    rectangle={"shapeOptions": {"color": "#f7b731", "weight": 3}},
    polyline={}, polygon={}, circle={}, marker={}, circlemarker={},
)

# ------------------------------------------------------------------------
# Status widgets
# ------------------------------------------------------------------------

# Message shown above the map indicating whether drawing is currently enabled.
zoom_label = widgets.HTML(
    value=f"<span style='color:#e67e22;font-weight:bold'>"
          f"Zoom in to level {MIN_DRAW_ZOOM}+ to enable drawing "
          f"(current: {DEFAULT_ZOOM})</span>"
)
bbox_info = widgets.HTML(value="<i>No bounding box drawn yet.</i>")

# ------------------------------------------------------------------------
# Zoom guard
# ------------------------------------------------------------------------

# Enable drawing only when the user is zoomed in far enough to make a reasonable selection.
def on_zoom_change(change):
    z = change["new"]
    state["current_zoom"] = z
    if z >= MIN_DRAW_ZOOM:
        if not state["draw_ctrl_on_map"]:
            m.add_control(draw_ctrl)
            state["draw_ctrl_on_map"] = True
        zoom_label.value = (
            f"<span style='color:#27ae60;font-weight:bold'>"
            f"Drawing enabled (zoom {z} ≥ {MIN_DRAW_ZOOM})"
            f" — draw a rectangle on the map</span>"
        )
    else:
        if state["draw_ctrl_on_map"]:
            m.remove_control(draw_ctrl)
            state["draw_ctrl_on_map"] = False
        zoom_label.value = (
            f"<span style='color:#e67e22;font-weight:bold'>"
            f"Zoom in to level {MIN_DRAW_ZOOM}+ to enable drawing "
            f"(current: {z})</span>"
        )

m.observe(on_zoom_change, names=["zoom"])

# ---------------------------------------------------------------------
# Draw event handler
# ---------------------------------------------------------------------

# Save the rectangle drawn by the user, convert it to a bounding box, and
# show the selected area on the map. The saved geometry remains available
# even if the user later zooms back out.
def on_draw(self, action, geo_json):
    if action != "created":
        return
    if geo_json.get("geometry", {}).get("type", "") not in ("Polygon", "Rectangle"):
        return

    bbox = geojson_rect_to_bbox(geo_json)
    state["drawn_bbox"]    = bbox

    w, s, e, n = bbox
    bbox_info.value = (
        f"<b>Bounding Box:</b> "
        f"W={w:.5f}  S={s:.5f}  E={e:.5f}  N={n:.5f}"
    )

    # Replace previous bbox highlight layer
    if state["bbox_layer"] is not None:
        try: m.remove_layer(state["bbox_layer"])
        except Exception: pass

    rect_layer = GeoJSON(
        data={"type": "Feature",
              "geometry": {"type": "Polygon",
                           "coordinates": [[[w,s],[e,s],[e,n],[w,n],[w,s]]]},
              "properties": {}},
        style={"color": "#f7b731", "weight": 3, "fillOpacity": 0.08},
        name="Search Area"
    )
    m.add_layer(rect_layer)
    state["bbox_layer"] = rect_layer

    print(f"Done. Bounding box saved: {[round(v, 5) for v in bbox]}")
    print("Run step 6 to search the STAC catalog.")

draw_ctrl.on_draw(on_draw)

# ---------------------------------------------------------------------
# Render interface
# ---------------------------------------------------------------------

# Display the step title, status message, interactive map, and current
# bounding box summary as a single notebook layout.
display(widgets.VBox([
    widgets.HTML("<h4 style='margin:4px 0'> Draw your Spatial area of Interest</h4>"),
    zoom_label,
    m,
    bbox_info,
]))

### Step 6 - Search Catalog and Select Dataset

In [149]:
# ---------------------------------------------------------------------
# Search STAC collections for datasets intersecting the drawn bbox
# ---------------------------------------------------------------------
if state["drawn_bbox"] is None:
    print("Draw a bounding box in Step 5 above first.")
else:
    bbox = state["drawn_bbox"]
    n = len(_collections_cache)
    print(f"Searching {n} collections")

    # Run parallel search with a live progress completed-count
    raw_hits = []
    completed = 0
    with ThreadPoolExecutor(max_workers=16) as pool:
        futures = {pool.submit(_check_collection, col, bbox): col for col in _collections_cache}
        for future in as_completed(futures):
            completed += 1
            raw_hits.extend(future.result())
            print(f"\r {completed}/{n} collections checked — {len(raw_hits)} raw hit(s) so far …", end="", flush=True)
            if len(raw_hits) >= MAX_RESULTS:
                for f in futures:
                    f.cancel()
                break

    print() # newline after progress tracking

    # Strictly filter hits to include collections that have a true COG/GeoTIFF asset
    hits = []
    for col, item in raw_hits:
        if any(is_tiff_asset(a) for a in item.assets.values()):
            hits.append((col, item))

    # Bound the results to MAX_RESULTS after the filter step
    hits = hits[:MAX_RESULTS]
    state["search_hits"] = hits

    if not hits:
        print("No native COG-formatted datasets intersect this bounding box.")
        print("Try drawing a larger bounding box over a data-rich area.")
    else:
        print(f"Done. Found {len(hits)} verified COG-formatted dataset(s):\n")
        for i, (col, item) in enumerate(hits):
            tiff_keys = [k for k, a in item.assets.items() if is_tiff_asset(a)]
            all_keys = list(item.assets.keys())
            print(f" [{i:2d}] {col.title or col.id}")
            print(f"      item id   : {item.id}")
            print(f"      item bbox : {[round(v,3) for v in item.bbox]}")
            print(f"      all assets: {all_keys}")
            print(f"      COG keys  : {tiff_keys}")
            print()

        # Build the drop down dataset selector from the filtered results only
        opts = [
            (f"[{i}] {col.title or col.id} ▸ {item.id}", i)
            for i, (col, item) in enumerate(hits)
        ]

        dataset_dd = widgets.Dropdown(
            options=opts,
            value=0,
            description='Dataset:',
            layout=widgets.Layout(width='100%')
        )

        def on_dropdown_change(change):
            if change['type'] == 'change' and change['name'] == 'value':
                idx = change['new']
                selected_col, selected_item = state["search_hits"][idx]
                state["selected_col"] = selected_col
                state["selected_item"] = selected_item

        dataset_dd.observe(on_dropdown_change, names=["value"])

        # Default initial selection to the first valid filtered hit
        col0, item0 = hits[0]
        state["selected_col"] = col0
        state["selected_item"] = item0

        

Searching 283 collections
 229/283 collections checked — 20 raw hit(s) so far …

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [163]:
display(widgets.VBox([
            widgets.HTML("<h4 style='margin:6px 0 4px'>Select a Dataset</h4>"),
            dataset_dd,
            widgets.HTML("<i>Run Step 7 to load its full metadata.</i>")
        ]))

### Step 7 - Dataset Metadata

In [164]:
# ---------------------------------------------------------------------
# Export selected dataset metadata to disk and show summary
# ---------------------------------------------------------------------
col  = state.get("selected_col")
item = state.get("selected_item")

if col is None or item is None:
    print("Run step 6 and select a dataset first.")
else:
    # Build metadata from the pystac objects already in memory
    meta_dict = col.to_dict()
    meta_dict["_selected_item"] = item.to_dict()

    ts        = datetime.now().strftime("%Y%m%d_%H%M%S")
    meta_path = OUTPUT_DIR / f"metadata_{col.id}_{ts}.json"
    meta_path.write_text(json.dumps(meta_dict, indent=2))
    state["metadata_path"] = meta_path

    # Print a compact, readable summary for quick inspection
    bar = "═" * 64
    print(bar)
    print(f"  COLLECTION  : {meta_dict.get('title', col.id)}")
    print(f"  ID          : {col.id}")
    desc = meta_dict.get("description", "")
    if desc:
        print(f"  DESCRIPTION : {desc[:200]}{'…' if len(desc)>200 else ''}")

    # Surface a few useful link types if present (about, license, citation)
    for lk in meta_dict.get("links", []):
        if lk.get("rel") in ("about", "license", "cite-as"):
            print(f"  {lk['rel'].upper():12}: {lk.get('href','')[:72]}")
    print(bar)
    print(f"  ITEM        : {item.id}")
    print(f"  ITEM BBOX   : {item.bbox}")
    print()

    print(f"\n Full metadata saved → {meta_path}")
    print("\n Run step 8 to generate the Color Hillshade.")

════════════════════════════════════════════════════════════════
  COLLECTION  : Yosemite Rim Fire lidar 2013
  ID          : OTSDEM.022017.26910.2
  DESCRIPTION : The Rim Fire started on August 17, 2013 in a remote area of the Stanislaus National Forest near the confluence of the Clavey and Tuolumne Rivers about 20 miles east of Sonora, California. The Rim Fire…
  ABOUT       : https://portal.opentopography.org/raster?opentopoID=OTSDEM.022017.26910.
  LICENSE     : https://creativecommons.org/licenses/by/4.0/
════════════════════════════════════════════════════════════════
  ITEM        : CA13_Ramirez_be
  ITEM BBOX   : [-120.29301453, 37.690729803, -119.62211742, 38.108940902]


 Full metadata saved → ot_stac_outputs/metadata_OTSDEM.022017.26910.2_20260721_141240.json

 Run step 8 to generate the Color Hillshade.


Step 8 - Generate Color Hillshade

In [165]:
item = state.get("selected_item")
col  = state.get("selected_col")
bbox = state.get("drawn_bbox")

if item is None or bbox is None:
    print("Complete steps 5–7 first.")
else:
    # Optional debugging block for asset inspection.
    # Uncomment if you need to diagnose which assets the item exposes.
    """
    print(f"Assets for item '{item.id}':")
    for k, a in item.assets.items():
        mt  = getattr(a, "media_type", "") or ""
        ef  = getattr(a, "extra_fields", {}) or {}
        t2  = ef.get("type", "") or ""
        detected = "✓ raster" if is_tiff_asset(a) else ""
        print(f"  [{k}]  media_type={mt!r}  ef.type={t2!r}  {detected}")
        print(f"    {a.href}")
    print()
    """
    # Choose the best raster asset using the 2-tier selection logic.
    # (see find_tiff_asset in step 4)
    asset, asset_bbox = find_tiff_asset(item, query_bbox=bbox)

    if asset is None:
        print("No usable raster asset found for this item.")
        print("Select a different dataset in step 6 and re-run step 7–8.")
    else:
        west, south, east, north = asset_bbox
        print(f" Selected asset href:")
        print(f" {asset.href}")

        print(f" bbox: {[round(v,4) for v in asset_bbox]}")
        print(f"\n Opening raster (may take a moment for remote COG files) …")

        try:
            # Inspect dataset metadata
            with rasterio.open(asset.href) as ds:
                print(f"   CRS    : {ds.crs}")
                print(f"   Size   : {ds.width} x {ds.height} px  (full dataset resolution)")
                print(f"   Bands  : {ds.count}")
                print(f"   Nodata : {ds.nodata}")
                print(f"   Bounds : {ds.bounds}")
                overviews = ds.overviews(1)
                print(f"   COG overviews: {overviews if overviews else 'none'}")

            # Read a downsampled preview clipped to the drawn AOI, so a huge
            # single-tile global COG (e.g. GEDTM) comes back as just the AOI
            # instead of the whole planet. rasterio requests only the AOI's
            # pixel window from the remote file (a windowed/ranged read).
            print(f"\n Reading AOI window at ≤{MAX_DISPLAY_PX}px (clipped + overview/resampled)")
            band, nodata, dx, dy, bounds, crs, shape = read_raster_safe(asset.href, bbox=bbox)
            print(f"   Read shape : {shape[1]} x {shape[0]} px (clipped to AOI)")
            print(f"   RAM usage  : ~{band.nbytes / 1e6:.1f} MB for this array")
            west, south, east, north = bounds.left, bounds.bottom, bounds.right, bounds.top

            # Render a terrain-colored hillshade image from the preview array.
            print(f"\n Rendering color hillshade …")
            img = make_color_hillshade_rgba(band, nodata=nodata, dx=dx, dy=dy)

            # Save the rendered preview to disk.
            ts      = datetime.now().strftime("%Y%m%d_%H%M%S")
            hs_path = OUTPUT_DIR / f"hillshade_{col.id}_{ts}.png"
            img.save(str(hs_path))
            state["hillshade_path"] = hs_path

            # Show the rendered preview inline in matplotlib.
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.imshow(img)
            ax.set_title(
                f"{col.title or col.id}\n"
                f"Elev range: {np.nanmin(band):.1f} – {np.nanmax(band):.1f} m  |  "
                f"{img.size[0]}x{img.size[1]} px",
                fontsize=10
            )
            ax.axis("off")
            plt.tight_layout()
            plt.show()

            # Replace any previous overlay and add the new image to the map (from step 5)
            if state["overlay_layer"] is not None:
                try: m.remove_layer(state["overlay_layer"])
                except Exception: pass

            overlay = ImageOverlay(
                url=pil_to_data_url(img),
                bounds=((south, west), (north, east)),
                opacity=0.85,
                name="Color Hillshade"
            )
            m.add_layer(overlay)
            m.center = ((south + north) / 2.0, (west + east) / 2.0)
            state["overlay_layer"] = overlay

            print(f"\n Done!")
            print(f"   Elevation range : {np.nanmin(band):.1f} – {np.nanmax(band):.1f} m")
            print(f"   Image size      : {img.size[0]}x{img.size[1]} px")
            print(f"   Saved           : {hs_path}")
            print("\n  Overlay added to the map — scroll up to step 5 to see it.")
            print("\n  Run step 9 to download all output files.")

        except Exception as ex:
            import traceback
            print(f"  Failed to open raster: {ex}")
            print(f"  Asset href: {asset.href}")
            print("\n Full traceback:")
            traceback.print_exc()
            print("\nTip: try selecting a different dataset in step 6 and re-running steps 7–8.")

 Selected asset href:
 https://opentopography.s3.sdsc.edu/raster/CA13_Ramirez/CA13_Ramirez_be/f738_4180.tif
 bbox: [-120.2991, 37.7281, -119.9517, 37.9164]

 Opening raster (may take a moment for remote COG files) …
   CRS    : EPSG:26910
   Size   : 30002 x 20002 px  (full dataset resolution)
   Bands  : 1
   Nodata : 1.701410009187828e+38
   Bounds : BoundingBox(left=737999.5, bottom=4179999.5, right=768001.5, top=4200001.5)
   COG overviews: none

 Reading AOI window at ≤1024px (clipped + overview/resampled)
  Failed to open raster: Bounds and transform are inconsistent
  Asset href: https://opentopography.s3.sdsc.edu/raster/CA13_Ramirez/CA13_Ramirez_be/f738_4180.tif

 Full traceback:

Tip: try selecting a different dataset in step 6 and re-running steps 7–8.


Traceback (most recent call last):
  File "/var/folders/4v/y_140j5x4bl_24qh3wzpjnp40000gn/T/ipykernel_17758/986880102.py", line 52, in <module>
    band, nodata, dx, dy, bounds, crs, shape = read_raster_safe(asset.href, bbox=bbox)
                                               ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/4v/y_140j5x4bl_24qh3wzpjnp40000gn/T/ipykernel_17758/2448160186.py", line 172, in read_raster_safe
    window = from_bounds(west, south, east, north, transform=full_tf)
  File "/Users/beckley/miniconda3/envs/Jupyter_TileIndex/lib/python3.13/site-packages/rasterio/windows.py", line 322, in from_bounds
    raise WindowError("Bounds and transform are inconsistent")
rasterio.errors.WindowError: Bounds and transform are inconsistent


### Step 9 - Download Output Files

In [ ]:
# ---------------------------------------------------------------------
# List and download generated output files
# ---------------------------------------------------------------------
files = sorted(OUTPUT_DIR.iterdir())

if not files:
    print(" No output files yet — run steps 7 and 8 first.")
else:
    print(f" {len(files)} file(s) in {OUTPUT_DIR}:\n")
    for fp in files:
        # Read the file and build a browser-download link using a data URL.
        size_kb = fp.stat().st_size / 1024
        data    = fp.read_bytes()
        b64     = base64.b64encode(data).decode()
        mime    = "image/png" if fp.suffix == ".png" else "application/json"
        href    = f"data:{mime};base64,{b64}"

        display(HTML(
            f'<a href="{href}" download="{fp.name}" '
            f'style="display:inline-block;margin:4px 0;padding:7px 16px;'
            f'background:#2980b9;color:white;border-radius:5px;'
            f'text-decoration:none;font-size:13px;font-family:sans-serif;">'
            f' {fp.name} ({size_kb:.1f} KB)</a>'
        ))
    print()
    print(" In Colab you can also click the Files icon in the left sidebar,")
    print(" navigate to ot_stac_outputs/, right-click a file → Download.")